# Day 03: Incremental Ingestion, Watermarking & Metadata Tracking
> **Part 3 of 10 in the PySpark Production Data Engineering Series**

[![PySpark](https://img.shields.io/badge/PySpark-3.5%2B%20%2F%204.0%2B-F26522?style=for-the-badge&logo=apachespark&logoColor=white)](https://spark.apache.org/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Status](https://img.shields.io/badge/Module-Day%2003%20of%2010-008080?style=for-the-badge)](../README.md)

---

## 🔄 Day 03: Incremental Ingestion, Watermarking & Metadata Tracking
### 🎯 Learning Objectives:
1. Transition from expensive **Full Table Scans** to resilient **Incremental Batch Ingestion**.
2. Build an external **Metadata State Store** (`metadata/processed_files.csv`) to track ingested files.
3. Implement file-arrival detection using path patterns and metadata differential matching.
4. Add pipeline audit watermarks (`ingestion_timestamp`) to trace data lineage.
5. Append new partitions into `bronze_incremental` without re-writing historical data.
6. Cascade incremental updates downstream through Silver and Gold layers.
---


### ⚙️ Initialize SparkSession for Day 03


In [77]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Day3_IncrementalIngestion")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [78]:
import os

os.makedirs("source", exist_ok=True)

print("Source directory created")

Source directory created


In [79]:
import pandas as pd

day1_data = {
    "order_id": [3001, 3002, 3003, 3004],
    "customer_id": [101, 102, 103, 104],
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor"],
    "quantity": [1, 2, 1, 1],
    "price": [55000, 800, 1500, 12000],
    "order_date": [
        "2026-09-27",
        "2026-09-27",
        "2026-09-27",
        "2026-09-27"
    ]
}

pd.DataFrame(day1_data).to_csv(
    "source/orders_2026_09_27.csv",
    index=False
)

print("Day 1 file created")

Day 1 file created


In [80]:
day2_data = {
    "order_id": [3005, 3006, 3007],
    "customer_id": [105, 106, 107],
    "product": ["Laptop", "Mouse", "Headphones"],
    "quantity": [2, 3, 1],
    "price": [55000, 800, 2500],
    "order_date": [
        "2026-09-28",
        "2026-09-28",
        "2026-09-28"
    ]
}

pd.DataFrame(day2_data).to_csv(
    "source/orders_2026_09_28.csv",
    index=False
)

print("Day 2 file created")

Day 2 file created


In [81]:
day3_data = {
    "order_id": [3008, 3009, 3010],
    "customer_id": [108, 109, 110],
    "product": ["Keyboard", "Laptop", "Monitor"],
    "quantity": [2, 1, 2],
    "price": [1500, 55000, 12000],
    "order_date": [
        "2026-09-29",
        "2026-09-29",
        "2026-09-29"
    ]
}

pd.DataFrame(day3_data).to_csv(
    "source/orders_2026_09_29.csv",
    index=False
)

print("Day 3 file created")

Day 3 file created


In [82]:
os.listdir("source")

['orders_2026_09_27.csv',
 'orders_2026_09_28.csv',
 'orders_2026_09_29.csv',
 'orders_2026_09_30.csv']

In [83]:
order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", StringType(), True)
])

In [84]:
spark.read.csv("source/*.csv")

DataFrame[_c0: string, _c1: string, _c2: string, _c3: string, _c4: string, _c5: string]

In [85]:
all_orders = (
    spark.read
    .option("header", True)
    .schema(order_schema)
    .csv("source/*.csv")
)

In [86]:
all_orders.show()

+--------+-----------+----------+--------+-------+----------+
|order_id|customer_id|   product|quantity|  price|order_date|
+--------+-----------+----------+--------+-------+----------+
|    3001|        101|    Laptop|       1|55000.0|2026-09-27|
|    3002|        102|     Mouse|       2|  800.0|2026-09-27|
|    3003|        103|  Keyboard|       1| 1500.0|2026-09-27|
|    3004|        104|   Monitor|       1|12000.0|2026-09-27|
|    3008|        108|  Keyboard|       2| 1500.0|2026-09-29|
|    3009|        109|    Laptop|       1|55000.0|2026-09-29|
|    3010|        110|   Monitor|       2|12000.0|2026-09-29|
|    3005|        105|    Laptop|       2|55000.0|2026-09-28|
|    3006|        106|     Mouse|       3|  800.0|2026-09-28|
|    3007|        107|Headphones|       1| 2500.0|2026-09-28|
|    3011|        111|     Mouse|       2|  800.0|2026-09-30|
|    3012|        112|    Laptop|       1|55000.0|2026-09-30|
+--------+-----------+----------+--------+-------+----------+



In [87]:
os.makedirs("metadata", exist_ok=True)

print("Metadata directory created")

Metadata directory created


In [88]:
processed_files = pd.DataFrame({
    "file_name": []
})

processed_files.to_csv(
    "metadata/processed_files.csv",
    index=False
)

In [89]:
pd.read_csv("metadata/processed_files.csv")

,file_name


In [90]:
import glob

source_files = glob.glob(
    "source/*.csv"
)

source_files

['source\\orders_2026_09_27.csv',
 'source\\orders_2026_09_28.csv',
 'source\\orders_2026_09_29.csv',
 'source\\orders_2026_09_30.csv']

In [91]:
source_file_names = [
    os.path.basename(file)
    for file in source_files
]

source_file_names

['orders_2026_09_27.csv',
 'orders_2026_09_28.csv',
 'orders_2026_09_29.csv',
 'orders_2026_09_30.csv']

In [92]:
processed_df = pd.read_csv(
    "metadata/processed_files.csv"
)

processed_files = set(
    processed_df["file_name"].tolist()
)

processed_files

set()

In [93]:
new_files = [
    file
    for file in source_file_names
    if file not in processed_files
]

new_files

['orders_2026_09_27.csv',
 'orders_2026_09_28.csv',
 'orders_2026_09_29.csv',
 'orders_2026_09_30.csv']

In [94]:
new_file_paths = [
    f"source/{file}"
    for file in new_files
]

new_orders = (
    spark.read
    .option("header", True)
    .schema(order_schema)
    .csv(new_file_paths)
)

In [95]:
new_orders.show()

+--------+-----------+----------+--------+-------+----------+
|order_id|customer_id|   product|quantity|  price|order_date|
+--------+-----------+----------+--------+-------+----------+
|    3001|        101|    Laptop|       1|55000.0|2026-09-27|
|    3002|        102|     Mouse|       2|  800.0|2026-09-27|
|    3003|        103|  Keyboard|       1| 1500.0|2026-09-27|
|    3004|        104|   Monitor|       1|12000.0|2026-09-27|
|    3008|        108|  Keyboard|       2| 1500.0|2026-09-29|
|    3009|        109|    Laptop|       1|55000.0|2026-09-29|
|    3010|        110|   Monitor|       2|12000.0|2026-09-29|
|    3005|        105|    Laptop|       2|55000.0|2026-09-28|
|    3006|        106|     Mouse|       3|  800.0|2026-09-28|
|    3007|        107|Headphones|       1| 2500.0|2026-09-28|
|    3011|        111|     Mouse|       2|  800.0|2026-09-30|
|    3012|        112|    Laptop|       1|55000.0|2026-09-30|
+--------+-----------+----------+--------+-------+----------+



In [96]:
new_orders.count()

12

In [97]:
new_orders = new_orders.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

In [98]:
new_orders.show()

+--------+-----------+----------+--------+-------+----------+--------------------+
|order_id|customer_id|   product|quantity|  price|order_date| ingestion_timestamp|
+--------+-----------+----------+--------+-------+----------+--------------------+
|    3001|        101|    Laptop|       1|55000.0|2026-09-27|2026-10-08 07:30:...|
|    3002|        102|     Mouse|       2|  800.0|2026-09-27|2026-10-08 07:30:...|
|    3003|        103|  Keyboard|       1| 1500.0|2026-09-27|2026-10-08 07:30:...|
|    3004|        104|   Monitor|       1|12000.0|2026-09-27|2026-10-08 07:30:...|
|    3008|        108|  Keyboard|       2| 1500.0|2026-09-29|2026-10-08 07:30:...|
|    3009|        109|    Laptop|       1|55000.0|2026-09-29|2026-10-08 07:30:...|
|    3010|        110|   Monitor|       2|12000.0|2026-09-29|2026-10-08 07:30:...|
|    3005|        105|    Laptop|       2|55000.0|2026-09-28|2026-10-08 07:30:...|
|    3006|        106|     Mouse|       3|  800.0|2026-09-28|2026-10-08 07:30:...|
|   

In [99]:
new_orders.write \
    .mode("append") \
    .parquet("bronze_incremental")

In [100]:
processed_df = pd.DataFrame({
    "file_name": new_files
})

processed_df.to_csv(
    "metadata/processed_files.csv",
    index=False
)

In [101]:
pd.read_csv(
    "metadata/processed_files.csv"
)

,file_name
0,orders_2026_09_27.csv
1,orders_2026_09_28.csv
2,orders_2026_09_29.csv
3,orders_2026_09_30.csv


In [102]:
day4_data = {
    "order_id": [3011, 3012],
    "customer_id": [111, 112],
    "product": ["Mouse", "Laptop"],
    "quantity": [2, 1],
    "price": [800, 55000],
    "order_date": [
        "2026-09-30",
        "2026-09-30"
    ]
}

pd.DataFrame(day4_data).to_csv(
    "source/orders_2026_09_30.csv",
    index=False
)

print("New file arrived!")

New file arrived!


In [103]:
source_files = glob.glob(
    "source/*.csv"
)

source_file_names = [
    os.path.basename(file)
    for file in source_files
]

processed_df = pd.read_csv(
    "metadata/processed_files.csv"
)

processed_files = set(
    processed_df["file_name"].tolist()
)

new_files = [
    file
    for file in source_file_names
    if file not in processed_files
]

new_files

[]

In [104]:
new_file_paths = [
    f"source/{file}"
    for file in new_files
]

new_orders = (
    spark.read
    .option("header", True)
    .schema(order_schema)
    .csv(new_file_paths)
)

In [105]:
new_orders.show()

+--------+-----------+-------+--------+-----+----------+
|order_id|customer_id|product|quantity|price|order_date|
+--------+-----------+-------+--------+-----+----------+
+--------+-----------+-------+--------+-----+----------+



In [106]:
new_orders = new_orders.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

new_orders.write \
    .mode("append") \
    .parquet("bronze_incremental")

In [107]:
old_processed = pd.read_csv(
    "metadata/processed_files.csv"
)

new_processed = pd.DataFrame({
    "file_name": new_files
})

updated_processed = pd.concat(
    [old_processed, new_processed],
    ignore_index=True
)

updated_processed.to_csv(
    "metadata/processed_files.csv",
    index=False
)

In [108]:
pd.read_csv(
    "metadata/processed_files.csv"
)

,file_name
0,orders_2026_09_27.csv
1,orders_2026_09_28.csv
2,orders_2026_09_29.csv
3,orders_2026_09_30.csv


In [109]:
source_files = glob.glob(
    "source/*.csv"
)

source_file_names = [
    os.path.basename(file)
    for file in source_files
]

processed_df = pd.read_csv(
    "metadata/processed_files.csv"
)

processed_files = set(
    processed_df["file_name"].tolist()
)

new_files = [
    file
    for file in source_file_names
    if file not in processed_files
]

print("New files:", new_files)

New files: []


In [110]:
bronze_df = spark.read.parquet(
    "bronze_incremental"
)

bronze_df.show()

+--------+-----------+--------+--------+-------+----------+--------------------+
|order_id|customer_id| product|quantity|  price|order_date| ingestion_timestamp|
+--------+-----------+--------+--------+-------+----------+--------------------+
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-08 07:30:...|
|    3002|        102|   Mouse|       2|  800.0|2026-09-27|2026-10-08 07:30:...|
|    3003|        103|Keyboard|       1| 1500.0|2026-09-27|2026-10-08 07:30:...|
|    3004|        104| Monitor|       1|12000.0|2026-09-27|2026-10-08 07:30:...|
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-05 08:03:...|
|    3002|        102|   Mouse|       2|  800.0|2026-09-27|2026-10-05 08:03:...|
|    3003|        103|Keyboard|       1| 1500.0|2026-09-27|2026-10-05 08:03:...|
|    3004|        104| Monitor|       1|12000.0|2026-09-27|2026-10-05 08:03:...|
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-03 07:56:...|
|    3002|        102|   Mou

In [111]:
silver_df = (
    bronze_df
    .withColumn(
        "order_date",
        to_date(col("order_date"))
    )
    .withColumn(
        "revenue",
        col("quantity") * col("price")
    )
)

In [112]:
gold_df = (
    silver_df
    .groupBy("product")
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("revenue").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
    .orderBy(desc("total_revenue"))
)

In [113]:
gold_df.show()

+----------+--------------+-------------+------------+
|   product|total_quantity|total_revenue|total_orders|
+----------+--------------+-------------+------------+
|    Laptop|            65|    3575000.0|           4|
|   Monitor|            39|     468000.0|           2|
|     Mouse|            91|      72800.0|           3|
|  Keyboard|            39|      58500.0|           2|
|Headphones|            13|      32500.0|           1|
+----------+--------------+-------------+------------+



---
### 🎓 Day 03 Complete!
Great job completing this module!
👉 **Next Up:** [**Day 04: Spark Performance Optimization, Partitioning & Broadcast Joins**](./Day_04_Performance_Tuning_Partitioning_and_Broadcast.ipynb)
